<a href="https://colab.research.google.com/github/zainabkashif1/IoT-Intrusion-Detection-ML/blob/main/notebooks/03_Eight_Class_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Import libraries
import pandas as pd
import numpy as np
import os

# Load the cleaned dataset
DATA_PATH = "/content/drive/MyDrive/shared iot/processed_data/cleaned_dataset.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

# Inspect all original attack labels
print("\nUnique Attack_Type labels:")
attack_labels = sorted(df["Attack_Type"].dropna().unique())

for label in attack_labels:
    print(label)

print("\nTotal unique attack labels:", len(attack_labels))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset shape: (730242, 41)

Columns:
['Header_Length', 'Protocol Type', 'Time_To_Live', 'Rate', 'fin_flag_number', 'syn_flag_number', 'rst_flag_number', 'psh_flag_number', 'ack_flag_number', 'ece_flag_number', 'cwr_flag_number', 'ack_count', 'syn_count', 'fin_count', 'rst_count', 'HTTP', 'HTTPS', 'DNS', 'Telnet', 'SMTP', 'SSH', 'IRC', 'TCP', 'UDP', 'DHCP', 'ARP', 'ICMP', 'IGMP', 'IPv', 'LLC', 'Tot sum', 'Min', 'Max', 'AVG', 'Std', 'Tot size', 'IAT', 'Number', 'Variance', 'Attack_Type', 'Binary_Label']

Unique Attack_Type labels:
Backdoor_Malware
Benign_Final
BrowserHijacking
CommandInjection
DDoS-ACK_Fragmentation
DDoS-HTTP_Flood
DDoS-ICMP_Flood
DDoS-ICMP_Fragmentation
DDoS-PSHACK_FLOOD
DDoS-RSTFINFLOOD
DDoS-SYN_Flood
DDoS-SlowLoris
DDoS-SynonymousIP_Flood
DDoS-TCP_Flood
DDoS-UDP_Flood
DDoS-UDP_Fragmentation
DNS_Spoofing
DictionaryBruteForce
DoS-HTTP_Flood
D

In [7]:
# Explicit mapping of all 34 original labels into 8 categories

attack_mapping = {
    "Benign_Final": "Benign",

    # DDoS
    "DDoS-ACK_Fragmentation": "DDoS",
    "DDoS-HTTP_Flood": "DDoS",
    "DDoS-ICMP_Flood": "DDoS",
    "DDoS-ICMP_Fragmentation": "DDoS",
    "DDoS-PSHACK_FLOOD": "DDoS",
    "DDoS-RSTFINFLOOD": "DDoS",
    "DDoS-SYN_Flood": "DDoS",
    "DDoS-SlowLoris": "DDoS",
    "DDoS-SynonymousIP_Flood": "DDoS",
    "DDoS-TCP_Flood": "DDoS",
    "DDoS-UDP_Flood": "DDoS",
    "DDoS-UDP_Fragmentation": "DDoS",

    # DoS
    "DoS-HTTP_Flood": "DoS",
    "DoS-SYN_Flood": "DoS",
    "DoS-TCP_Flood": "DoS",
    "DoS-UDP_Flood": "DoS",

    # Reconnaissance
    "Recon-HostDiscovery": "Recon",
    "Recon-OSScan": "Recon",
    "Recon-PingSweep": "Recon",
    "Recon-PortScan": "Recon",
    "VulnerabilityScan": "Recon",

    # Web-based
    "Backdoor_Malware": "Web-based",
    "BrowserHijacking": "Web-based",
    "CommandInjection": "Web-based",
    "SqlInjection": "Web-based",
    "Uploading_Attack": "Web-based",
    "XSS": "Web-based",

    # Brute Force
    "DictionaryBruteForce": "Brute Force",

    # Spoofing
    "DNS_Spoofing": "Spoofing",
    "MITM-ArpSpoofing": "Spoofing",

    # Mirai
    "Mirai-greeth_flood": "Mirai",
    "Mirai-greip_flood": "Mirai",
    "Mirai-udpplain": "Mirai"
}

# Apply mapping
df["Attack_Category"] = df["Attack_Type"].map(attack_mapping)

# Check for unmapped labels
unmapped = df[df["Attack_Category"].isna()]["Attack_Type"].value_counts()

if len(unmapped) > 0:
    print("ERROR: Unmapped labels found:")
    print(unmapped)
else:
    print("All original labels successfully mapped!")

# Display the 8-class distribution
print("\nClass distribution:")
print(df["Attack_Category"].value_counts())

print("\nNumber of classes:", df["Attack_Category"].nunique())

All original labels successfully mapped!

Class distribution:
Attack_Category
DDoS           489701
DoS            135318
Mirai           56142
Benign          23455
Recon           14752
Spoofing        10065
Web-based         530
Brute Force       279
Name: count, dtype: int64

Number of classes: 8


In [8]:
# Separate input features and 8-class target

feature_columns = [
    col for col in df.columns
    if col not in ["Attack_Type", "Binary_Label", "Attack_Category"]
]

X = df[feature_columns].copy()
y = df["Attack_Category"].copy()

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

# Generate a hash for each feature row
feature_hash = pd.util.hash_pandas_object(
    X,
    index=False
)

# Combine feature hashes with their corresponding labels
hash_labels = pd.DataFrame({
    "Feature_Hash": feature_hash,
    "Attack_Category": y
})

# Count how many different 8-class labels each feature group has
labels_per_group = hash_labels.groupby(
    "Feature_Hash"
)["Attack_Category"].nunique()

# Identify groups with conflicting labels
conflicting_groups = labels_per_group[
    labels_per_group > 1
].index

print("\nNumber of conflicting feature groups:",
      len(conflicting_groups))

# Identify all rows belonging to conflicting groups
conflicting_mask = feature_hash.isin(conflicting_groups)

print("Number of conflicting rows:",
      conflicting_mask.sum())

# Remove conflicting groups
df_8class = df.loc[~conflicting_mask].copy()

print("\nDataset shape after removing conflicts:",
      df_8class.shape)

print("\nClass distribution after conflict removal:")
print(df_8class["Attack_Category"].value_counts())

Feature shape: (730242, 39)
Target shape: (730242,)

Number of conflicting feature groups: 25592
Number of conflicting rows: 55486

Dataset shape after removing conflicts: (674756, 42)

Class distribution after conflict removal:
Attack_Category
DDoS           459857
DoS            109764
Mirai           56142
Benign          23430
Recon           14735
Spoofing        10023
Web-based         526
Brute Force       279
Name: count, dtype: int64


In [9]:
from sklearn.model_selection import StratifiedGroupKFold

# Prepare features and labels after conflict removal
X_8 = df_8class[feature_columns].copy()
y_8 = df_8class["Attack_Category"].copy()

# Generate feature hashes again for the filtered dataset
groups_8 = pd.util.hash_pandas_object(
    X_8,
    index=False
)

# Create group-aware stratified split
sgkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

train_idx, test_idx = next(
    sgkf.split(X_8, y_8, groups=groups_8)
)

# Split the data
X_train_8 = X_8.iloc[train_idx].copy()
X_test_8 = X_8.iloc[test_idx].copy()

y_train_8 = y_8.iloc[train_idx].copy()
y_test_8 = y_8.iloc[test_idx].copy()

# Display shapes
print("Training features:", X_train_8.shape)
print("Testing features:", X_test_8.shape)

# Check class distribution
print("\nTraining class distribution:")
print(y_train_8.value_counts())

print("\nTesting class distribution:")
print(y_test_8.value_counts())

# Verify that no feature groups overlap
train_groups = set(groups_8.iloc[train_idx])
test_groups = set(groups_8.iloc[test_idx])

overlap = train_groups.intersection(test_groups)

print("\nOverlapping feature groups:", len(overlap))

Training features: (539807, 39)
Testing features: (134949, 39)

Training class distribution:
Attack_Category
DDoS           368132
DoS             87783
Mirai           44716
Benign          18715
Recon           11824
Spoofing         7989
Web-based         424
Brute Force       224
Name: count, dtype: int64

Testing class distribution:
Attack_Category
DDoS           91725
DoS            21981
Mirai          11426
Benign          4715
Recon           2911
Spoofing        2034
Web-based        102
Brute Force       55
Name: count, dtype: int64

Overlapping feature groups: 0


In [10]:
from sklearn.preprocessing import StandardScaler

scaler_8 = StandardScaler()

X_train_8_scaled = scaler_8.fit_transform(X_train_8)
X_test_8_scaled = scaler_8.transform(X_test_8)

print("Training:", X_train_8_scaled.shape)
print("Testing:", X_test_8_scaled.shape)

Training: (539807, 39)
Testing: (134949, 39)


In [11]:
from sklearn.ensemble import RandomForestClassifier

feature_selector_8 = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

feature_selector_8.fit(X_train_8_scaled, y_train_8)

importance_8 = pd.DataFrame({
    "Feature": feature_columns,
    "Importance": feature_selector_8.feature_importances_
})

importance_8 = importance_8.sort_values(
    by="Importance",
    ascending=False
).reset_index(drop=True)

display(importance_8)

top_15_features_8 = importance_8.head(15)["Feature"].tolist()

print("\nTop 15 features:")
print(top_15_features_8)

,Feature,Importance
0,Rate,0.148420
1,IAT,0.146907
2,Tot size,0.084471
3,AVG,0.073560
4,Tot sum,0.062029
5,Max,0.055934
6,Protocol Type,0.043292
7,Number,0.035412
8,Header_Length,0.033608
9,Min,0.033325



Top 15 features:
['Rate', 'IAT', 'Tot size', 'AVG', 'Tot sum', 'Max', 'Protocol Type', 'Number', 'Header_Length', 'Min', 'Variance', 'Std', 'Time_To_Live', 'ack_flag_number', 'ack_count']


In [12]:
top_15_indices_8 = [
    feature_columns.index(feature)
    for feature in top_15_features_8
]

# 39 features
X_train_39_8 = X_train_8_scaled
X_test_39_8 = X_test_8_scaled

# 15 features
X_train_15_8 = X_train_8_scaled[:, top_15_indices_8]
X_test_15_8 = X_test_8_scaled[:, top_15_indices_8]

print("39-feature training:", X_train_39_8.shape)
print("39-feature testing:", X_test_39_8.shape)

print("15-feature training:", X_train_15_8.shape)
print("15-feature testing:", X_test_15_8.shape)

39-feature training: (539807, 39)
39-feature testing: (134949, 39)
15-feature training: (539807, 15)
15-feature testing: (134949, 15)


In [13]:
from sklearn.preprocessing import LabelEncoder

label_encoder_8 = LabelEncoder()

y_train_8_encoded = label_encoder_8.fit_transform(y_train_8)
y_test_8_encoded = label_encoder_8.transform(y_test_8)

print("Class encoding:")

for i, class_name in enumerate(label_encoder_8.classes_):
    print(f"{class_name}: {i}")

print("\nEncoded training labels:", y_train_8_encoded.shape)
print("Encoded testing labels:", y_test_8_encoded.shape)

Class encoding:
Benign: 0
Brute Force: 1
DDoS: 2
DoS: 3
Mirai: 4
Recon: 5
Spoofing: 6
Web-based: 7

Encoded training labels: (539807,)
Encoded testing labels: (134949,)


In [14]:
import joblib
import os

SAVE_PATH = "/content/drive/MyDrive/shared iot/8class_saved"
os.makedirs(SAVE_PATH, exist_ok=True)

joblib.dump({
    "X_train_39": X_train_39_8,
    "X_test_39": X_test_39_8,
    "X_train_15": X_train_15_8,
    "X_test_15": X_test_15_8,
    "y_train": y_train_8,
    "y_test": y_test_8,
    "y_train_encoded": y_train_8_encoded,
    "y_test_encoded": y_test_8_encoded,
    "top_15_features": top_15_features_8,
    "label_encoder": label_encoder_8
}, SAVE_PATH + "/eight_class_data.joblib")

print("8-class data successfully saved!")

8-class data successfully saved!


In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Train the baseline using all 39 features
baseline_39 = DummyClassifier(strategy="most_frequent")
baseline_39.fit(X_train_39_8, y_train_8)

# Predict on test data
baseline_pred_39 = baseline_39.predict(X_test_39_8)

# Evaluate
baseline_accuracy = accuracy_score(y_test_8, baseline_pred_39)

baseline_macro_f1 = f1_score(
    y_test_8,
    baseline_pred_39,
    average="macro",
    zero_division=0
)

print("Majority Class Baseline")
print("-----------------------")
print("Accuracy:", baseline_accuracy)
print("Macro-F1:", baseline_macro_f1)

print("\nClassification Report:")
print(classification_report(
    y_test_8,
    baseline_pred_39,
    labels=label_encoder_8.classes_,
    zero_division=0
))

Majority Class Baseline
-----------------------
Accuracy: 0.6797012204610631
Macro-F1: 0.10116400645861458

Classification Report:
              precision    recall  f1-score   support

      Benign       0.00      0.00      0.00      4715
 Brute Force       0.00      0.00      0.00        55
        DDoS       0.68      1.00      0.81     91725
         DoS       0.00      0.00      0.00     21981
       Mirai       0.00      0.00      0.00     11426
       Recon       0.00      0.00      0.00      2911
    Spoofing       0.00      0.00      0.00      2034
   Web-based       0.00      0.00      0.00       102

    accuracy                           0.68    134949
   macro avg       0.08      0.12      0.10    134949
weighted avg       0.46      0.68      0.55    134949



In [ ]:
import os

RESULTS_FOLDER = "/content/drive/MyDrive/shared iot/results"
os.makedirs(RESULTS_FOLDER, exist_ok=True)

baseline_results_8 = pd.DataFrame([{
    "Model": "Majority Baseline",
    "Features": 39,
    "Balancing": "None",
    "Accuracy": baseline_accuracy,
    "Macro-F1": baseline_macro_f1
}])

baseline_results_8.to_csv(
    os.path.join(RESULTS_FOLDER, "eight_class_results.csv"),
    index=False
)

display(baseline_results_8)

,Model,Features,Balancing,Accuracy,Macro-F1
0,Majority Baseline,39,None,0.679701,0.101164


## logistic regression
using 15 and 39 features

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

# Logistic Regression using all 39 features
lr_39_8 = LogisticRegression(max_iter=1000, random_state=42)
lr_39_8.fit(X_train_39_8, y_train_8)

pred_lr_39_8 = lr_39_8.predict(X_test_39_8)

acc_lr_39_8 = accuracy_score(y_test_8, pred_lr_39_8)
f1_lr_39_8 = f1_score(y_test_8, pred_lr_39_8, average="macro", zero_division=0)


# Logistic Regression using top 15 features
lr_15_8 = LogisticRegression(max_iter=1000, random_state=42)
lr_15_8.fit(X_train_15_8, y_train_8)

pred_lr_15_8 = lr_15_8.predict(X_test_15_8)

acc_lr_15_8 = accuracy_score(y_test_8, pred_lr_15_8)
f1_lr_15_8 = f1_score(y_test_8, pred_lr_15_8, average="macro", zero_division=0)


# Display results
lr_results_8 = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Features": 39,
        "Balancing": "None",
        "Accuracy": acc_lr_39_8,
        "Macro-F1": f1_lr_39_8
    },
    {
        "Model": "Logistic Regression",
        "Features": 15,
        "Balancing": "None",
        "Accuracy": acc_lr_15_8,
        "Macro-F1": f1_lr_15_8
    }
])

display(lr_results_8)

,Model,Features,Balancing,Accuracy,Macro-F1
0,Logistic Regression,39,None,0.840762,0.556466
1,Logistic Regression,15,None,0.824823,0.467454


## decision trees
using 15 and 39 features


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score

# Decision Tree: 39 features
dt_39_8 = DecisionTreeClassifier(random_state=42)
dt_39_8.fit(X_train_39_8, y_train_8)

pred_dt_39_8 = dt_39_8.predict(X_test_39_8)

acc_dt_39_8 = accuracy_score(y_test_8, pred_dt_39_8)
f1_dt_39_8 = f1_score(y_test_8, pred_dt_39_8, average="macro", zero_division=0)


# Decision Tree: 15 features
dt_15_8 = DecisionTreeClassifier(random_state=42)
dt_15_8.fit(X_train_15_8, y_train_8)

pred_dt_15_8 = dt_15_8.predict(X_test_15_8)

acc_dt_15_8 = accuracy_score(y_test_8, pred_dt_15_8)
f1_dt_15_8 = f1_score(y_test_8, pred_dt_15_8, average="macro", zero_division=0)


# Display results
dt_results_8 = pd.DataFrame([
    {
        "Model": "Decision Tree",
        "Features": 39,
        "Balancing": "None",
        "Accuracy": acc_dt_39_8,
        "Macro-F1": f1_dt_39_8
    },
    {
        "Model": "Decision Tree",
        "Features": 15,
        "Balancing": "None",
        "Accuracy": acc_dt_15_8,
        "Macro-F1": f1_dt_15_8
    }
])

display(dt_results_8)

,Model,Features,Balancing,Accuracy,Macro-F1
0,Decision Tree,39,None,0.831129,0.620353
1,Decision Tree,15,None,0.813626,0.578932


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

# Random Forest: 39 features
rf_39_8 = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_39_8.fit(X_train_39_8, y_train_8)
pred_rf_39_8 = rf_39_8.predict(X_test_39_8)

acc_rf_39_8 = accuracy_score(y_test_8, pred_rf_39_8)
f1_rf_39_8 = f1_score(
    y_test_8, pred_rf_39_8,
    average="macro", zero_division=0
)


# Random Forest: 15 features
rf_15_8 = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_15_8.fit(X_train_15_8, y_train_8)
pred_rf_15_8 = rf_15_8.predict(X_test_15_8)

acc_rf_15_8 = accuracy_score(y_test_8, pred_rf_15_8)
f1_rf_15_8 = f1_score(
    y_test_8, pred_rf_15_8,
    average="macro", zero_division=0
)


# Display results
rf_results_8 = pd.DataFrame([
    {
        "Model": "Random Forest",
        "Features": 39,
        "Balancing": "None",
        "Accuracy": acc_rf_39_8,
        "Macro-F1": f1_rf_39_8
    },
    {
        "Model": "Random Forest",
        "Features": 15,
        "Balancing": "None",
        "Accuracy": acc_rf_15_8,
        "Macro-F1": f1_rf_15_8
    }
])

display(rf_results_8)

,Model,Features,Balancing,Accuracy,Macro-F1
0,Random Forest,39,None,0.854478,0.648228
1,Random Forest,15,None,0.835012,0.616246


In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score

# XGBoost: 39 features
xgb_39_8 = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    objective="multi:softprob",
    num_class=8,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

xgb_39_8.fit(X_train_39_8, y_train_8_encoded)

pred_xgb_39_8 = xgb_39_8.predict(X_test_39_8)

acc_xgb_39_8 = accuracy_score(y_test_8_encoded, pred_xgb_39_8)
f1_xgb_39_8 = f1_score(
    y_test_8_encoded, pred_xgb_39_8,
    average="macro", zero_division=0
)


# XGBoost: 15 features
xgb_15_8 = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    objective="multi:softprob",
    num_class=8,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

xgb_15_8.fit(X_train_15_8, y_train_8_encoded)

pred_xgb_15_8 = xgb_15_8.predict(X_test_15_8)

acc_xgb_15_8 = accuracy_score(y_test_8_encoded, pred_xgb_15_8)
f1_xgb_15_8 = f1_score(
    y_test_8_encoded, pred_xgb_15_8,
    average="macro", zero_division=0
)


# Display results
xgb_results_8 = pd.DataFrame([
    {
        "Model": "XGBoost",
        "Features": 39,
        "Balancing": "None",
        "Accuracy": acc_xgb_39_8,
        "Macro-F1": f1_xgb_39_8
    },
    {
        "Model": "XGBoost",
        "Features": 15,
        "Balancing": "None",
        "Accuracy": acc_xgb_15_8,
        "Macro-F1": f1_xgb_15_8
    }
])

display(xgb_results_8)

,Model,Features,Balancing,Accuracy,Macro-F1
0,XGBoost,39,None,0.869929,0.666103
1,XGBoost,15,None,0.858613,0.612296


In [15]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score

# MLP: 39 features
mlp_39_8 = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation="relu",
    solver="adam",
    max_iter=30,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1
)

mlp_39_8.fit(X_train_39_8, y_train_8_encoded)

pred_mlp_39_8 = mlp_39_8.predict(X_test_39_8)

acc_mlp_39_8 = accuracy_score(y_test_8_encoded, pred_mlp_39_8)
f1_mlp_39_8 = f1_score(
    y_test_8_encoded, pred_mlp_39_8,
    average="macro", zero_division=0
)


# MLP: 15 features
mlp_15_8 = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation="relu",
    solver="adam",
    max_iter=30,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1
)

mlp_15_8.fit(X_train_15_8, y_train_8_encoded)

pred_mlp_15_8 = mlp_15_8.predict(X_test_15_8)

acc_mlp_15_8 = accuracy_score(y_test_8_encoded, pred_mlp_15_8)
f1_mlp_15_8 = f1_score(
    y_test_8_encoded, pred_mlp_15_8,
    average="macro", zero_division=0
)


# Display results
mlp_results_8 = pd.DataFrame([
    {
        "Model": "MLP",
        "Features": 39,
        "Balancing": "None",
        "Accuracy": acc_mlp_39_8,
        "Macro-F1": f1_mlp_39_8
    },
    {
        "Model": "MLP",
        "Features": 15,
        "Balancing": "None",
        "Accuracy": acc_mlp_15_8,
        "Macro-F1": f1_mlp_15_8
    }
])

display(mlp_results_8)

/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (30) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (30) reached and the optimization hasn't converged yet.
  warnings.warn(


,Model,Features,Balancing,Accuracy,Macro-F1
0,MLP,39,None,0.866023,0.636156
1,MLP,15,None,0.854627,0.580288


In [6]:
import pandas as pd
import os

results_8 = pd.DataFrame([
    ["Majority Baseline", 39, "None", 0.679701, 0.101164],
    ["Logistic Regression", 39, "None", 0.840762, 0.556466],
    ["Logistic Regression", 15, "None", 0.824823, 0.467454],
    ["Decision Tree", 39, "None", 0.831129, 0.620353],
    ["Decision Tree", 15, "None", 0.813626, 0.578932],
    ["Random Forest", 39, "None", 0.854478, 0.648228],
    ["Random Forest", 15, "None", 0.835012, 0.616246],
    ["XGBoost", 39, "None", 0.869929, 0.666103],
    ["XGBoost", 15, "None", 0.858613, 0.612296]
], columns=["Model", "Features", "Balancing", "Accuracy", "Macro-F1"])

RESULTS_PATH = "/content/drive/MyDrive/shared iot/results"
os.makedirs(RESULTS_PATH, exist_ok=True)

results_8.to_csv(
    RESULTS_PATH + "/eight_class_results.csv",
    index=False
)

display(results_8)

print("Results saved to Google Drive!")

,Model,Features,Balancing,Accuracy,Macro-F1
0,Majority Baseline,39,None,0.679701,0.101164
1,Logistic Regression,39,None,0.840762,0.556466
2,Logistic Regression,15,None,0.824823,0.467454
3,Decision Tree,39,None,0.831129,0.620353
4,Decision Tree,15,None,0.813626,0.578932
5,Random Forest,39,None,0.854478,0.648228
6,Random Forest,15,None,0.835012,0.616246
7,XGBoost,39,None,0.869929,0.666103
8,XGBoost,15,None,0.858613,0.612296


Results saved to Google Drive!


In [16]:
import os
import joblib
import pandas as pd

# Save all model results
results_8 = pd.DataFrame([
    ["Majority Baseline", 39, "None", 0.679701, 0.101164],
    ["Logistic Regression", 39, "None", 0.840762, 0.556466],
    ["Logistic Regression", 15, "None", 0.824823, 0.467454],
    ["Decision Tree", 39, "None", 0.831129, 0.620353],
    ["Decision Tree", 15, "None", 0.813626, 0.578932],
    ["Random Forest", 39, "None", 0.854478, 0.648228],
    ["Random Forest", 15, "None", 0.835012, 0.616246],
    ["XGBoost", 39, "None", 0.869929, 0.666103],
    ["XGBoost", 15, "None", 0.858613, 0.612296],
    ["MLP", 39, "None", 0.866023, 0.636156],
    ["MLP", 15, "None", 0.854627, 0.580288]
], columns=["Model", "Features", "Balancing", "Accuracy", "Macro-F1"])

RESULTS_PATH = "/content/drive/MyDrive/shared iot/results"
SAVE_PATH = "/content/drive/MyDrive/shared iot/8class_saved"

os.makedirs(RESULTS_PATH, exist_ok=True)
os.makedirs(SAVE_PATH, exist_ok=True)

results_8.to_csv(
    RESULTS_PATH + "/eight_class_results.csv",
    index=False
)

# Save prepared data
joblib.dump({
    "X_train_39": X_train_39_8,
    "X_test_39": X_test_39_8,
    "X_train_15": X_train_15_8,
    "X_test_15": X_test_15_8,
    "y_train": y_train_8,
    "y_test": y_test_8,
    "y_train_encoded": y_train_8_encoded,
    "y_test_encoded": y_test_8_encoded,
    "top_15_features": top_15_features_8,
    "label_encoder": label_encoder_8
}, SAVE_PATH + "/eight_class_data.joblib")

print("Results and prepared data saved successfully!")

Results and prepared data saved successfully!


In [19]:
from sklearn.ensemble import RandomForestClassifier

rf_39_8 = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_39_8.fit(X_train_39_8, y_train_8)

pred_rf_39_8 = rf_39_8.predict(X_test_39_8)

print("Random Forest predictions generated!")

Random Forest predictions generated!


In [20]:
from xgboost import XGBClassifier

xgb_39_8 = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    objective="multi:softprob",
    num_class=8,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

xgb_39_8.fit(X_train_39_8, y_train_8_encoded)

pred_xgb_39_8 = xgb_39_8.predict(X_test_39_8)

print("XGBoost predictions generated!")

XGBoost predictions generated!


In [22]:
import joblib
import os

SAVE_PATH = "/content/drive/MyDrive/shared iot/8class_saved"
os.makedirs(SAVE_PATH, exist_ok=True)

joblib.dump(
    {
        "random_forest": rf_39_8,
        "xgboost": xgb_39_8
    },
    SAVE_PATH + "/trained_models.joblib"
)

print("Both models saved!")

Both models saved!


## per-class classification report
for best performing models

In [23]:
from sklearn.metrics import classification_report

# Random Forest report
rf_report = classification_report(
    y_test_8,
    pred_rf_39_8,
    labels=label_encoder_8.classes_,
    output_dict=True,
    zero_division=0
)

# XGBoost report
xgb_report = classification_report(
    y_test_8_encoded,
    pred_xgb_39_8,
    labels=range(8),
    target_names=label_encoder_8.classes_,
    output_dict=True,
    zero_division=0
)

rf_report_df = pd.DataFrame(rf_report).transpose()
xgb_report_df = pd.DataFrame(xgb_report).transpose()

display(rf_report_df)
display(xgb_report_df)

rf_report_df.to_csv(RESULTS_PATH + "/rf_8class_per_class_report.csv")
xgb_report_df.to_csv(RESULTS_PATH + "/xgb_8class_per_class_report.csv")

print("Classification reports saved!")

,precision,recall,f1-score,support
Benign,0.812783,0.876564,0.843469,4715.000000
Brute Force,0.800000,0.072727,0.133333,55.000000
DDoS,0.892890,0.915639,0.904121,91725.000000
DoS,0.606769,0.543196,0.573225,21981.000000
Mirai,0.999122,0.996062,0.997590,11426.000000
Recon,0.744402,0.742357,0.743378,2911.000000
Spoofing,0.935573,0.835300,0.882597,2034.000000
Web-based,0.666667,0.058824,0.108108,102.000000
accuracy,0.854478,0.854478,0.854478,0.854478
macro avg,0.807276,0.630084,0.648228,134949.000000


,precision,recall,f1-score,support
Benign,0.805165,0.886108,0.843700,4715.000000
Brute Force,0.714286,0.181818,0.289855,55.000000
DDoS,0.873645,0.968253,0.918519,91725.000000
DoS,0.758966,0.416860,0.538145,21981.000000
Mirai,0.999211,0.996937,0.998072,11426.000000
Recon,0.763335,0.742357,0.752699,2911.000000
Spoofing,0.942568,0.823009,0.878740,2034.000000
Web-based,0.750000,0.058824,0.109091,102.000000
accuracy,0.869929,0.869929,0.869929,0.869929
macro avg,0.825897,0.634271,0.666103,134949.000000


Classification reports saved!


## random forest with class weighting
for 15 and 39 features

In [24]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

# Random Forest with class weighting: 39 features
rf_weighted_39_8 = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_weighted_39_8.fit(X_train_39_8, y_train_8)

pred_rf_weighted_39_8 = rf_weighted_39_8.predict(X_test_39_8)


# Random Forest with class weighting: 15 features
rf_weighted_15_8 = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_weighted_15_8.fit(X_train_15_8, y_train_8)

pred_rf_weighted_15_8 = rf_weighted_15_8.predict(X_test_15_8)


# Results
rf_weighted_results_8 = pd.DataFrame([
    {
        "Model": "Random Forest",
        "Features": 39,
        "Balancing": "Class Weight",
        "Accuracy": accuracy_score(y_test_8, pred_rf_weighted_39_8),
        "Macro-F1": f1_score(y_test_8, pred_rf_weighted_39_8, average="macro", zero_division=0)
    },
    {
        "Model": "Random Forest",
        "Features": 15,
        "Balancing": "Class Weight",
        "Accuracy": accuracy_score(y_test_8, pred_rf_weighted_15_8),
        "Macro-F1": f1_score(y_test_8, pred_rf_weighted_15_8, average="macro", zero_division=0)
    }
])

display(rf_weighted_results_8)

,Model,Features,Balancing,Accuracy,Macro-F1
0,Random Forest,39,Class Weight,0.854834,0.668270
1,Random Forest,15,Class Weight,0.826268,0.609141


In [26]:
from sklearn.metrics import classification_report
import pandas as pd

# Per-class report for weighted Random Forest (39 features)
weighted_rf_report_39 = pd.DataFrame(
    classification_report(
        y_test_8,
        pred_rf_weighted_39_8,
        target_names=label_encoder_8.classes_,
        output_dict=True,
        zero_division=0
    )
).transpose()

display(weighted_rf_report_39)

# Save report
weighted_rf_report_39.to_csv(
    "/content/drive/MyDrive/shared iot/results/rf_weighted_8class_per_class_report.csv"
)

,precision,recall,f1-score,support
Benign,0.806576,0.884411,0.843703,4715.000000
Brute Force,0.588235,0.181818,0.277778,55.000000
DDoS,0.891321,0.918179,0.904551,91725.000000
DoS,0.610355,0.534689,0.570022,21981.000000
Mirai,0.999210,0.995712,0.997457,11426.000000
Recon,0.754597,0.733081,0.743684,2911.000000
Spoofing,0.940390,0.829892,0.881692,2034.000000
Web-based,0.875000,0.068627,0.127273,102.000000
accuracy,0.854834,0.854834,0.854834,0.854834
macro avg,0.808211,0.643301,0.668270,134949.000000


## weighted xgboost
with 15 and 39 features

In [27]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.utils.class_weight import compute_sample_weight

# Calculate balanced sample weights
sample_weights_8 = compute_sample_weight(
    class_weight="balanced",
    y=y_train_8
)

# XGBoost with class weighting: 39 features
xgb_weighted_39_8 = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    objective="multi:softprob",
    num_class=8,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

xgb_weighted_39_8.fit(
    X_train_39_8,
    y_train_8_encoded,
    sample_weight=sample_weights_8
)

pred_xgb_weighted_39_8 = xgb_weighted_39_8.predict(X_test_39_8)


# XGBoost with class weighting: 15 features
xgb_weighted_15_8 = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    objective="multi:softprob",
    num_class=8,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

xgb_weighted_15_8.fit(
    X_train_15_8,
    y_train_8_encoded,
    sample_weight=sample_weights_8
)

pred_xgb_weighted_15_8 = xgb_weighted_15_8.predict(X_test_15_8)


# Results
xgb_weighted_results_8 = pd.DataFrame([
    {
        "Model": "XGBoost",
        "Features": 39,
        "Balancing": "Class Weight",
        "Accuracy": accuracy_score(y_test_8_encoded, pred_xgb_weighted_39_8),
        "Macro-F1": f1_score(y_test_8_encoded, pred_xgb_weighted_39_8, average="macro", zero_division=0)
    },
    {
        "Model": "XGBoost",
        "Features": 15,
        "Balancing": "Class Weight",
        "Accuracy": accuracy_score(y_test_8_encoded, pred_xgb_weighted_15_8),
        "Macro-F1": f1_score(y_test_8_encoded, pred_xgb_weighted_15_8, average="macro", zero_division=0)
    }
])

display(xgb_weighted_results_8)

,Model,Features,Balancing,Accuracy,Macro-F1
0,XGBoost,39,Class Weight,0.788609,0.636065
1,XGBoost,15,Class Weight,0.696345,0.585762


In [28]:
xgb_weighted_results_8.to_csv(
    "/content/drive/MyDrive/shared iot/results/xgb_weighted_8class_results.csv",
    index=False
)

print("Weighted XGBoost results saved!")

Weighted XGBoost results saved!


In [29]:
weighted_xgb_report_39 = pd.DataFrame(
    classification_report(
        y_test_8_encoded,
        pred_xgb_weighted_39_8,
        target_names=label_encoder_8.classes_,
        output_dict=True,
        zero_division=0
    )
).transpose()

display(weighted_xgb_report_39)

weighted_xgb_report_39.to_csv(
    "/content/drive/MyDrive/shared iot/results/xgb_weighted_8class_per_class_report.csv"
)

,precision,recall,f1-score,support
Benign,0.859506,0.782397,0.819141,4715.000000
Brute Force,0.075758,0.363636,0.125392,55.000000
DDoS,0.967895,0.739853,0.838648,91725.000000
DoS,0.453120,0.897684,0.602246,21981.000000
Mirai,0.993983,0.997549,0.995763,11426.000000
Recon,0.741419,0.667812,0.702693,2911.000000
Spoofing,0.925846,0.847099,0.884724,2034.000000
Web-based,0.067775,0.519608,0.119910,102.000000
accuracy,0.788609,0.788609,0.788609,0.788609
macro avg,0.635663,0.726955,0.636065,134949.000000


In [30]:
rf_weighted_results_8.to_csv(
    "/content/drive/MyDrive/shared iot/results/rf_weighted_8class_results.csv",
    index=False
)